# Week 95, Day 5: Responding to review

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w095d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
comments = ["Why `>` rather than `>=` here?", "nit: two blank lines before the class", "LGTM, thanks!",
            "Please add a test for a width of 0."]
print([c.endswith("?") for c in comments])

> **Key idea.** Sort comments: suggestions (apply them), requests (do them or explain), questions (answer, and perhaps add a code comment), nits (usually just do them), approvals (thank). Reply to every comment briefly and specifically, without defensive words; push new commits and re-request review. Track threads: resolved, waiting on you, waiting on the reviewer. When main moves on, merge or rebase, resolve conflicts with the base in mind, and test again.

### Exercise: What does the comment ask for?

Write `classify(comment)`, checking in this order on the stripped comment: `"suggestion"` if it contains three backticks followed by `suggestion`; `"nit"` if it starts with `nit` or contains `(nit)` (ignoring case); `"approval"` if it contains `lgtm` or `looks good` (ignoring case); `"question"` if it ends with `?` or its first word (ignoring case) is one of `why`, `what`, `how`, `could`, `should`, `would`, `is`, `are`, `does`, `do`, `can`; otherwise `"request"`.

In [ ]:
import re

def classify(comment):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("FENCE = \"`\" * 3\n\ndef test_kinds():\n    \"sort before replying\"\n    assert classify(\"Why `>` rather than `>=` here?\") == \"question\"\n    assert classify(\"Could you explain the loop. I'm not sure.\") == \"question\"\n    assert classify(\"nit: two blank lines before the class\") == \"nit\"\n    assert classify(\"Rename this (nit), it's fine either way.\") == \"nit\"\n    assert classify(\"LGTM, thanks!\") == \"approval\" and classify(\"This looks good to me.\") == \"approval\"\n    assert classify(f\"{FENCE}suggestion\\n        if length + extra + len(placeholder) > width:\\n{FENCE}\") == \"suggestion\"\n    assert classify(\"Please add a test for a width of 0.\") == \"request\"")

### Exercise: Applying a suggestion

Write `apply_suggestion(source, start, end, replacement)`: replace lines `start` to `end` of `source` (counting from 1, both included) with the lines of `replacement`, keeping a final newline if the source had one. Raise `ValueError("lines out of range")` unless `1 <= start <= end <= number of lines`.

In [ ]:
def apply_suggestion(source, start, end, replacement):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("SOURCE = \"def f(x):\\n    if x >= 18:\\n        return 1\\n    return 0\\n\"\n\ndef test_apply():\n    \"exactly the lines suggested\"\n    assert apply_suggestion(SOURCE, 2, 2, \"    if x > 18:\") == \"def f(x):\\n    if x > 18:\\n        return 1\\n    return 0\\n\"\n    assert apply_suggestion(SOURCE, 2, 3, \"    if x > 18:\\n        return 2\") == \"def f(x):\\n    if x > 18:\\n        return 2\\n    return 0\\n\"\n    assert apply_suggestion(\"a\\nb\", 2, 2, \"c\") == \"a\\nc\"\n    for start, end in [(0, 1), (3, 2), (4, 5)]:\n        try:\n            apply_suggestion(SOURCE, start, end, \"x\")\n        except ValueError as e:\n            assert str(e) == \"lines out of range\"\n        else:\n            raise AssertionError((start, end))")

### Exercise: Who is it waiting on?

Write `thread_status(threads, me)`: each thread is a dict with `id`, `resolved` and `comments` (a list of `(author, text)`). Return a dict with `resolved`, `waiting on you` and `waiting on reviewer`, each a list of thread ids in the order given: resolved threads; unresolved threads whose last comment is by someone other than `me`; and unresolved threads whose last comment is by `me`.

In [ ]:
def thread_status(threads, me):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("THREADS = [\n    {\"id\": \"T1\", \"resolved\": True, \"comments\": [(\"rev\", \"nit: blank lines\"), (\"ada\", \"Done, thanks.\")]},\n    {\"id\": \"T2\", \"resolved\": False, \"comments\": [(\"rev\", \"Why > rather than >=?\")]},\n    {\"id\": \"T3\", \"resolved\": False, \"comments\": [(\"rev\", \"Please add a test for width 0.\"), (\"ada\", \"Added in 3f2a1c9.\"), (\"rev\", \"Thanks, and width 1?\")]},\n    {\"id\": \"T4\", \"resolved\": False, \"comments\": [(\"rev2\", \"Should the docs mention this?\"), (\"ada\", \"I think not; it was always documented. Happy to add a line.\")]},\n]\n\ndef test_status():\n    \"two waiting on you\"\n    assert thread_status(THREADS, \"ada\") == {\"resolved\": [\"T1\"], \"waiting on you\": [\"T2\", \"T3\"], \"waiting on reviewer\": [\"T4\"]}\n    assert thread_status([], \"ada\") == {\"resolved\": [], \"waiting on you\": [], \"waiting on reviewer\": []}")

### Exercise: Replies that keep things moving

Write `reply_problems(reply)`: report, in this order, `"sounds defensive: WORD"` for each of `obviously`, `just`, `actually`, `you're wrong` and `not my problem` found in the reply (whole words or phrases, ignoring case), and `"say what changed"` if it contains none of `done`, `added`, `changed`, `fixed`, `updated`, `pushed`, `removed` or `renamed` (whole words, ignoring case) and doesn't contain a `?`.

In [ ]:
import re

def reply_problems(reply):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_replies():\n    \"short, specific, grateful\"\n    assert reply_problems(\"Added test_width_zero in 3f2a1c9, thanks!\") == []\n    assert reply_problems(\"Good question: would a comment on the loop help?\") == []\n    assert reply_problems(\"Obviously it's > because exact fits fit. Just read the issue.\") == [\n        \"sounds defensive: obviously\", \"sounds defensive: just\", \"say what changed\"]\n    assert reply_problems(\"Actually, done.\") == [\"sounds defensive: actually\"]\n    assert reply_problems(\"I adjusted it, justly.\") == [\"say what changed\"]")

### Exercise: A three-way merge

Write `merge3(base, ours, theirs)`: three lists of lines of the same length (the common ancestor, your branch, and `main`). For each position: if ours and theirs agree, take it; if only one side changed it from the base, take that side; otherwise write a conflict: `<<<<<<< ours`, our line, `=======`, their line, `>>>>>>> theirs`. Return `(lines, conflicts)`.

In [ ]:
def merge3(base, ours, theirs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_merge():\n    \"only real conflicts need you\"\n    base = [\"for word in words:\", \"    if total >= width:\", \"        break\", \"    kept.append(word)\"]\n    ours = [\"for word in words:\", \"    if total > width:\", \"        break\", \"    kept.append(word)\"]\n    theirs = [\"for word in words:\", \"    if total >= width:\", \"        break\", \"    kept.append(word)  # keep it\"]\n    assert merge3(base, ours, theirs) == ([\"for word in words:\", \"    if total > width:\", \"        break\", \"    kept.append(word)  # keep it\"], 0)\n    theirs2 = [\"for word in words:\", \"    if total >= max_width:\", \"        break\", \"    kept.append(word)\"]\n    lines, conflicts = merge3(base, ours, theirs2)\n    assert conflicts == 1 and lines[1:6] == [\"<<<<<<< ours\", \"    if total > width:\", \"=======\", \"    if total >= max_width:\", \">>>>>>> theirs\"]")

### Bug hunt: The suggestion applied one line low

Line numbers in a review count from 1, but this slices as if they counted from 0, so it replaces the wrong line and prints `['def f(x):', '    if x >= 18:', '    if x > 18:']`. Fix the slice so it prints `['def f(x):', '    if x > 18:', '        return 1']`.

In [ ]:
lines = ["def f(x):", "    if x >= 18:", "        return 1"]
start = end = 2
lines[start:end + 1] = ["    if x > 18:"]
print(lines)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"counting from 1\"\n    expect_output(\"['def f(x):', '    if x > 18:', '        return 1']\")")

### Bug hunt: A question taken as a request

The question word is checked with its capital letter, so `"why"` never matches `"Why"`, and this prints `request`. Compare in lower case so it prints `question`.

In [ ]:
comment = "Why not compare before adding the space. I may be missing something"
first = comment.split()[0]
print("question" if comment.endswith("?") or first in ("why", "what", "how") else "request")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"ignore case\"\n    expect_output(\"question\")")

### Puzzle: Sorting the threads

Arrange `thread_status`'s loop. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # what LGTM stands for (four words, lower case)
q = ___   # classify("nit: rename length to used")
r = ___   # who a thread is waiting on when the reviewer commented last ("you" or "reviewer")
s = ___   # True or False: force-push after review unless the project asks you not to
t = ___   # the marker line that starts a conflict block (seven characters and a word)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"looks good to me\" and q == \"nit\" and r == \"you\", \"p, q and r\"\n    assert s is False and t == \"<<<<<<< ours\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: one round of review

Write `review_round(threads, me, pushed)` with `classify` (given). `pushed` is the set of thread ids your latest push addressed. For each thread in order: skip it if resolved; if you commented last, the line is `"ID: waiting on reviewer"`; otherwise classify the **last** comment and write `"ID [KIND]: ACTION"`, where ACTION is `"thank them"` for an approval, `"answer the question"` for a question, `"reply: done"` for other kinds if the id is in `pushed`, or `"change needed"` if not. Return `(lines, verdict)`: `verdict` is `"re-request review"` if no thread needs a change, otherwise `"push changes for: "` and those ids joined by `", "`.

In [ ]:
import re

QUESTION_WORDS = ("why", "what", "how", "could", "should", "would", "is", "are", "does", "do", "can")

def classify(comment):
    text = comment.strip()
    lower = text.lower()
    if "`" * 3 + "suggestion" in text:
        return "suggestion"
    if lower.startswith("nit") or "(nit)" in lower:
        return "nit"
    if "lgtm" in lower or "looks good" in lower:
        return "approval"
    first = re.findall(r"[a-z']+", lower)[:1]
    if text.endswith("?") or (first and first[0] in QUESTION_WORDS):
        return "question"
    return "request"

def review_round(threads, me, pushed):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("FENCE = \"`\" * 3\nTHREADS = [\n    {\"id\": \"T1\", \"resolved\": True, \"comments\": [(\"rev\", \"nit: blank lines\"), (\"ada\", \"Done, thanks.\")]},\n    {\"id\": \"T2\", \"resolved\": False, \"comments\": [(\"rev\", \"Why > rather than >=?\")]},\n    {\"id\": \"T3\", \"resolved\": False, \"comments\": [(\"rev\", \"Please add a test for a width of 0.\")]},\n    {\"id\": \"T4\", \"resolved\": False, \"comments\": [(\"rev2\", \"Should the docs mention this?\"), (\"ada\", \"I think not. Happy to add a line?\")]},\n    {\"id\": \"T5\", \"resolved\": False, \"comments\": [(\"rev\", f\"{FENCE}suggestion\\n    # gh-77: an exact fit is kept\\n{FENCE}\")]},\n    {\"id\": \"T6\", \"resolved\": False, \"comments\": [(\"rev\", \"nit: two blank lines before the class\")]},\n    {\"id\": \"T7\", \"resolved\": False, \"comments\": [(\"rev3\", \"LGTM once the test is in.\")]},\n]\n\ndef test_round():\n    \"what to do next\"\n    lines, verdict = review_round(THREADS, \"ada\", {\"T3\", \"T5\"})\n    assert lines == [\"T2 [question]: answer the question\", \"T3 [request]: reply: done\", \"T4: waiting on reviewer\",\n                     \"T5 [suggestion]: reply: done\", \"T6 [nit]: change needed\", \"T7 [approval]: thank them\"]\n    assert verdict == \"push changes for: T6\"\n    assert review_round(THREADS, \"ada\", {\"T3\", \"T5\", \"T6\"})[1] == \"re-request review\"")

## Warm-up

Four comments, four different things to do: answer a question, make a small optional change, say thank you, and add a test. Treating them all alike (or all as criticism) is the commonest mistake; sorting them first makes the reply easy.

## Kinds of comment

- A **suggestion** comes with the exact change, in a block GitHub can apply with one click (a fenced block marked `suggestion`). If you agree, apply it.
- A **request** asks for a change ("please add a test for width 0"). Make it, or explain politely why not.
- A **question** ("why `>` rather than `>=`?") wants an answer, not necessarily a change. Often the answer belongs in a code comment or the commit message too, so the next reader doesn't need to ask.
- A **nit** (short for nitpick) is small and optional: style, naming. Usually just do it.
- An **approval** ("LGTM", looks good to me) needs only thanks.

Disagreement is fine: say what you think and why, briefly, then accept the decision. Core developers know the project's history and its users; sometimes the reason isn't obvious until they explain it.

## Answering well

Reply to every comment, even if only "Done, thanks". Keep replies short and specific: what you changed and where ("Added `test_width_zero` in the latest push"). Avoid words that sound defensive or dismissive ("obviously", "just", "actually", "that's not my problem"): text loses tone, and a sentence that sounds neutral to you can read as irritated. When you've addressed everything, push the changes (new commits, not a force-push, unless the project asks: reviewers can then see what changed since they looked) and re-request review.

## Keeping track

A pull request with fifteen comments across six threads is easy to lose track of. A thread is **resolved** when the conversation is finished; otherwise it's **waiting on you** (a reviewer spoke last) or **waiting on the reviewer** (you did). Go through every thread waiting on you before re-requesting review. Leave resolving to the reviewer when the project prefers that.

## When main moves on

While you waited, other changes merged. If one touched the same lines, GitHub reports a **conflict**. Update your branch (`git merge main`, or `git rebase main` if the project prefers), and Git marks each conflict in the file between `<<<<<<<`, `=======` and `>>>>>>>`. A **three-way merge** compares both versions with their common ancestor (the base): a line only one side changed takes that side's version; a line both changed differently is a conflict for you to resolve by hand, keeping both intentions. Then run the tests again.

## Practice

## Quiz

## Daily challenge

## Your turn

1. **Sort the comments** on your pull request (or, while you wait, on a merged CPython pull request you admire) into the five kinds.
2. **Answer every one**: apply suggestions and nits, make or politely discuss requests, answer questions (adding a code comment where it would help), and thank approvals.
3. **Update your branch** if `main` has moved, resolving any conflict and running the tests again.
4. **Push, then re-request review**, and wait. If a week passes with no response, a single polite comment asking whether anything else is needed is fine.

## Before you go

Tomorrow reviews the week, from a long bug report to a merged fix.